In [2]:
# ---------- CELL 1: find the project folder and create the structure ----------
import os, sys, re, zipfile, shutil
from pathlib import Path
import pandas as pd

ZIP_NAME = "netflix.zip"
MARKER = "PROJECT_ROOT.marker"

def find_upwards(filename):
    """Start in the folder this notebook runs in, then go UP one level at a time."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / filename).exists():
            return folder / filename
    return None

def find_in_drive(filename):
    """Colab only: mount Google Drive and search 3 levels deep for the file."""
    from google.colab import drive
    drive.mount("/content/drive")
    for pattern in (
        filename,
        f"*/{filename}",
        f"*/*/{filename}",
        f"*/*/*/{filename}"
    ):
        hits = sorted(Path("/content/drive/MyDrive").glob(pattern))
        if hits:
            return hits[0]
    return None

on_colab = "google.colab" in sys.modules

# 1) Is the project already set up?
found = find_upwards(MARKER) or (
    find_in_drive(MARKER) if on_colab else None
)

if found:
    ROOT = found.parent
else:
    # 2) First time: the folder containing netflix.zip becomes the project root
    zip_found = find_upwards(ZIP_NAME) or (
        find_in_drive(ZIP_NAME) if on_colab else None
    )

    if zip_found is None:
        raise FileNotFoundError(
            "Could not find netflix.zip. Upload it into your project folder first."
        )

    ROOT = zip_found.parent

FOLDERS = [
    "data/raw",
    "data/clean",
    "data/features",
    "data/model_outputs",
    "data/sql_exports",
    "data/bi_ready",
    "database",
    "notebooks",
    "src",
    "models",
    "reports/figures",
    "docs/project_briefs",
    "powerbi",
]

for folder in FOLDERS:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

(ROOT / MARKER).write_text(
    "This empty file marks the top folder of the project. Do not delete it.\n"
)

os.chdir(ROOT)

print("Project root is:", ROOT)

Mounted at /content/drive
Project root is: /content/drive/MyDrive/Internmo/netflix_churn_project


In [3]:
# ---------- CELL 2: extract the zip and rename files ----------
RAW = ROOT / "data" / "raw"
DOCS = ROOT / "docs"

# keep the original zip safely inside data/raw
zip_in_root = ROOT / ZIP_NAME
zip_in_raw = RAW / ZIP_NAME

if zip_in_root.exists():
    shutil.move(str(zip_in_root), str(zip_in_raw))

with zipfile.ZipFile(zip_in_raw) as z:
    for member in z.namelist():
        original_name = Path(member).name
        clean_name = re.sub(r"\s*\(\d+\)", "", original_name)

        if clean_name.lower().endswith(".csv"):
            target = RAW / clean_name
        elif clean_name.lower().endswith(".docx"):
            target = DOCS / "project_briefs" / clean_name
        else:
            continue

        target.write_bytes(z.read(member))
        print(f"{original_name:45s} -> {target.relative_to(ROOT)}")

churn_labels.csv                              -> data/raw/churn_labels.csv
content.csv                                   -> data/raw/content.csv
customer_feedback (2).csv                     -> data/raw/customer_feedback.csv
customers (3).csv                             -> data/raw/customers.csv
Netflix_Database_Schema.docx                  -> docs/project_briefs/Netflix_Database_Schema.docx
Netflix_Dataset_Dictionary.docx               -> docs/project_briefs/Netflix_Dataset_Dictionary.docx
Netflix_SQL_Business_Questions.docx           -> docs/project_briefs/Netflix_SQL_Business_Questions.docx
payments (2).csv                              -> data/raw/payments.csv
PRD_Netflix_Customer_Intelligence_Copilot.docx -> docs/project_briefs/PRD_Netflix_Customer_Intelligence_Copilot.docx
subscription_plans.csv                        -> data/raw/subscription_plans.csv
subscriptions (1).csv                         -> data/raw/subscriptions.csv
support_tickets (1).csv                       -> data/

In [4]:
# ---------- CELL 3: save the standard Cell 0 into docs/cell0_paths.py ----------
CELL0 = '''# ---------- CELL 0: locate the project (same in every notebook) ----------
import os, sys
from pathlib import Path

def find_root(marker="PROJECT_ROOT.marker"):
    here = Path.cwd().resolve()                      # start where this notebook runs
    for folder in [here, *here.parents]:             # go UP one level at a time
        if (folder / marker).exists():
            return folder
    if "google.colab" in sys.modules:                # Colab: look inside Google Drive
        from google.colab import drive
        drive.mount("/content/drive")
        for pattern in (marker, f"*/{marker}", f"*/*/{marker}", f"*/*/*/{marker}"):
            hits = sorted(Path("/content/drive/MyDrive").glob(pattern))
            if hits:
                return hits[0].parent
    raise FileNotFoundError("PROJECT_ROOT.marker not found - finish Day 1 first.")

ROOT = find_root()
os.chdir(ROOT / "notebooks")
RAW, CLEAN = ROOT/"data"/"raw", ROOT/"data"/"clean"
FEATURES, MODEL_OUT = ROOT/"data"/"features", ROOT/"data"/"model_outputs"
SQL_EXPORTS, BI_READY = ROOT/"data"/"sql_exports", ROOT/"data"/"bi_ready"
MODELS, REPORTS, DOCS = ROOT/"models", ROOT/"reports", ROOT/"docs"
FIGURES = REPORTS/"figures"
sys.path.append(str(ROOT/"src"))
print("Project root:", ROOT)
'''
(DOCS / "cell0_paths.py").write_text(CELL0)

# test it: run the saved file exactly like a later notebook will
exec(CELL0)
print("Cell 0 works. RAW folder =", RAW.relative_to(ROOT))

Project root: /content/drive/MyDrive/Internmo/netflix_churn_project
Cell 0 works. RAW folder = data/raw


In [5]:
# ---------- CELL 4: audit every raw table ----------
raw_tables = {
    path.stem: pd.read_csv(path)
    for path in sorted(RAW.glob("*.csv"))
}

table_rows = []

for name, df in raw_tables.items():
    table_rows.append({
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "fully_duplicated_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
    })

audit_tables = pd.DataFrame(table_rows)

audit_tables.to_csv(
    DOCS / "data_audit_tables.csv",
    index=False
)

audit_tables

,table,rows,columns,fully_duplicated_rows,missing_cells
0,churn_labels,8000,4,0,12018
1,content,500,9,0,0
2,customer_feedback,5046,6,0,0
3,customers,8048,12,48,9517
4,payments,110735,8,0,214264
5,subscription_plans,3,6,0,0
6,subscriptions,8875,10,0,21768
7,support_tickets,6120,11,0,3056
8,viewing_activity,66422,9,0,1195


In [6]:
# ---------- CELL 5: missing values per column ----------
column_rows = []

for name, df in raw_tables.items():
    for col in df.columns:
        missing = int(df[col].isna().sum())

        column_rows.append({
            "table": name,
            "column": col,
            "dtype": str(df[col].dtype),
            "missing": missing,
            "missing_pct": round(100 * missing / len(df), 2),
            "unique_values": int(df[col].nunique()),
        })

audit_columns = pd.DataFrame(column_rows)

audit_columns.to_csv(
    DOCS / "data_audit_columns.csv",
    index=False
)

# Show only columns that actually have missing values
audit_columns[
    audit_columns["missing"] > 0
].sort_values(
    "missing_pct",
    ascending=False
)

,table,column,dtype,missing,missing_pct,unique_values
33,payments,subscription_id,float64,110735,100.00,0
25,customers,state_or_region,float64,8048,100.00,0
38,payments,failed_payment_reason,object,103529,93.49,6
49,subscriptions,subscription_end_date,object,8000,90.14,670
53,subscriptions,cancellation_reason,object,6884,77.57,11
52,subscriptions,cancellation_date,object,6884,77.57,271
2,churn_labels,churn_date,object,6009,75.11,271
3,churn_labels,churn_reason,object,6009,75.11,11
62,support_tickets,resolution_time_hours,float64,1528,24.97,667
63,support_tickets,customer_satisfaction_score,float64,1528,24.97,5


In [7]:
# ---------- CELL 6: verify the known data problems ----------
customers = raw_tables["customers"]
viewing   = raw_tables["viewing_activity"]
tickets   = raw_tables["support_tickets"]
payments  = raw_tables["payments"]
feedback  = raw_tables["customer_feedback"]
churn     = raw_tables["churn_labels"]

print(
    "1) customers: duplicated customer_id rows      :",
    customers["customer_id"].duplicated().sum()
)

print(
    "2) customers: different spellings of country   :",
    customers["country"].nunique(),
    "(should be 20)"
)

print(
    "3) customers: impossible ages (<10 or >100)    :",
    ((customers["age"] < 10) | (customers["age"] > 100)).sum()
)

print(
    "4) viewing  : different spellings of device    :",
    viewing["device_type"].nunique(),
    "(should be 5)"
)

print(
    "5) viewing  : sessions longer than 480 minutes :",
    (viewing["watch_duration_minutes"] > 480).sum()
)

print(
    "6) tickets  : dates that cannot be parsed      :",
    pd.to_datetime(
        tickets["ticket_date"],
        errors="coerce"
    ).isna().sum()
)

print(
    "7) payments : rows with a subscription_id      :",
    payments["subscription_id"].notna().sum(),
    "(all blank)"
)

print(
    "8) payments : amounts equal to 0               :",
    (payments["amount"] == 0).sum()
)

print(
    "9) feedback : rows in each sentiment label     :",
    feedback["sentiment_label"].value_counts().to_dict()
)

print(
    "10) churn   : overall churn rate               :",
    round(churn["churned"].mean() * 100, 1),
    "%"
)

1) customers: duplicated customer_id rows      : 48
2) customers: different spellings of country   : 83 (should be 20)
3) customers: impossible ages (<10 or >100)    : 21
4) viewing  : different spellings of device    : 23 (should be 5)
5) viewing  : sessions longer than 480 minutes : 161
6) tickets  : dates that cannot be parsed      : 61
7) payments : rows with a subscription_id      : 0 (all blank)
8) payments : amounts equal to 0               : 250
9) feedback : rows in each sentiment label     : {'Positive': 1978, 'Neutral': 1606, 'Negative': 1462}
10) churn   : overall churn rate               : 24.9 %


In [11]:
# ---------- CELL 7: what period does each table cover? ----------

coverage = {
    "customers registered": (
        customers["registration_date"].min(),
        customers["registration_date"].max()
    ),
    "viewing sessions": (
        viewing["viewing_date"].min(),
        viewing["viewing_date"].max()
    ),
    "payments": (
        payments["payment_date"].min(),
        payments["payment_date"].max()
    ),
    "churn dates": (
        pd.to_datetime(churn["churn_date"], errors="coerce").min(),
        pd.to_datetime(churn["churn_date"], errors="coerce").max()
    ),
}

for name, (first, last) in coverage.items():
    print(f"{name:22s} {first}  ->  {last}")

customers registered   2019-01-01  ->  2026-05-31
viewing sessions       2025-07-01  ->  2026-06-30
payments               2019-01-01  ->  2026-05-31
churn dates            2025-10-03 00:00:00  ->  2026-06-30 00:00:00


In [12]:
# ---------- CELL 8: start the project log ----------

log_path = DOCS / "project_log.md"

if not log_path.exists():
    log_path.write_text(
        "# Project Log\n\n"
        "One entry per day. Write what you did and why.\n\n"
    )

entry = """## Day 1 - Setup and audit

- Created the project folder structure and PROJECT_ROOT.marker.
- Extracted 9 CSVs into data/raw with clean filenames.
- Extracted 4 project documents into docs/project_briefs.
- Created reusable path helper: docs/cell0_paths.py.
- Created audit files: docs/data_audit_tables.csv and docs/data_audit_columns.csv.
- Main activity period: 2025-07-01 to 2026-06-30.
- Project as-of date: 2026-06-30.

### Key data-quality findings
- customers: 48 duplicated rows/customer IDs.
- customers: 83 country labels versus approximately 20 expected countries.
- customers: 21 impossible ages.
- customers.state_or_region is completely missing.
- viewing_activity: 23 device labels versus 5 expected device types.
- viewing_activity: 161 sessions exceed 480 minutes.
- support_tickets: 61 dates could not be parsed.
- payments.subscription_id is completely blank across 110,735 rows.
- payments: 250 zero-value payment records.
- subscriptions: many end/cancellation fields are missing, likely reflecting active subscriptions.
- churn_labels: churn date/reason fields are missing for many non-churned customers.
- Overall churn rate: 24.9%.

### Project-document notes
- False negatives are more costly, so identifying true churn-risk customers is especially important.
- The project documents describe intentional data-quality issues including missing values, duplicate customers, inconsistent country/device labels, malformed dates, and extreme outliers.
- Churn is influenced by engagement, payment failures, support experience, plan type, and tenure.
- The dataset dictionary documents table relationships and expected cleaning approaches.
- The database schema documents table structures, relationships, and recommended load order.
- The SQL business-questions document provides the questions to answer during the SQL analysis phase.

Day 1 is an audit only. No raw data was modified or cleaned.
"""

with open(log_path, "a") as f:
    f.write(entry + "\n")

print(log_path.read_text())

# Project Log

One entry per day. Write what you did and why.

## Day 1 - Setup and audit

- Created the project folder structure and PROJECT_ROOT.marker.
- Extracted 9 CSVs into data/raw with clean filenames.
- Extracted 4 project documents into docs/project_briefs.
- Created reusable path helper: docs/cell0_paths.py.
- Created audit files: docs/data_audit_tables.csv and docs/data_audit_columns.csv.
- Main activity period: 2025-07-01 to 2026-06-30.
- Project as-of date: 2026-06-30.

### Key data-quality findings
- customers: 48 duplicated rows/customer IDs.
- customers: 83 country labels versus approximately 20 expected countries.
- customers: 21 impossible ages.
- customers.state_or_region is completely missing.
- viewing_activity: 23 device labels versus 5 expected device types.
- viewing_activity: 161 sessions exceed 480 minutes.
- support_tickets: 61 dates could not be parsed.
- payments.subscription_id is completely blank across 110,735 rows.
- payments: 250 zero-value payment re

In [14]:
# ---------- CELL 9: final Day 1 folder check ----------

folders_to_check = [
    ROOT,
    ROOT / "data",
    ROOT / "data" / "raw",
    ROOT / "docs",
    ROOT / "docs" / "project_briefs",
    ROOT / "notebooks",
]

for folder in folders_to_check:
    status = "OK" if folder.exists() else "MISSING"
    print(f"{folder.relative_to(ROOT)} -> {status}")

print("\nRaw CSV files:")
for f in sorted((ROOT / "data" / "raw").glob("*.csv")):
    print(" -", f.name)

print("\nAudit/project files:")
for f in sorted((ROOT / "docs").glob("*")):
    if f.is_file():
        print(" -", f.name)

. -> OK
data -> OK
data/raw -> OK
docs -> OK
docs/project_briefs -> OK
notebooks -> OK

Raw CSV files:
 - churn_labels.csv
 - content.csv
 - customer_feedback.csv
 - customers.csv
 - payments.csv
 - subscription_plans.csv
 - subscriptions.csv
 - support_tickets.csv
 - viewing_activity.csv

Audit/project files:
 - cell0_paths.py
 - data_audit_columns.csv
 - data_audit_tables.csv
 - project_log.md
